In [129]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import Adam
from network import Network

In [130]:
train_df = pd.read_csv("datasets//train_energy_data.csv")
test_df = pd.read_csv("datasets//test_energy_data.csv")

In [131]:
feature_headings = ["Square Footage", "Number of Occupants", "Appliances Used", "Average Temperature"]
target_heading = "Energy Consumption"

X_train = train_df[feature_headings].to_numpy()
Y_train = train_df[target_heading].to_numpy()

X_test = test_df[feature_headings].to_numpy()
Y_test = test_df[target_heading].to_numpy()

X_mean = np.concatenate([X_train, X_test], axis=0).mean(axis=0)
X_std = np.concatenate([X_train, X_test], axis=0).std(axis=0)
Y_mean = np.concatenate([Y_train, Y_test], axis=0).mean(axis=0)
Y_std = np.concatenate([Y_train, Y_test], axis=0).std(axis=0)

x_train = (X_train - X_mean)/X_std
y_train = (Y_train - Y_mean)/Y_std
x_test = (X_test - X_mean)/X_std
y_test = (Y_test - Y_mean)/Y_std

In [132]:
x_train = torch.tensor(x_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
x_test = torch.tensor(x_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
def wTw(model):

    wTw = (model.fc_1.weight.data * model.fc_1.weight.data).sum() + \
              (model.fc_1.bias.data * model.fc_1.bias.data).sum() + \
              (model.fc_2.weight.data * model.fc_2.weight.data).sum() + \
              (model.fc_2.bias.data * model.fc_2.bias.data).sum() + \
              (model.fc_3.weight.data * model.fc_3.weight.data).sum() + \
              (model.fc_3.bias.data * model.fc_3.bias.data).sum()

    return wTw

def log_posterior_weights(model, predicts, target, alpha, beta):

    wTw = wTw(model)
    TSE = ((predicts - target)**2).sum()
    return -0.5 * (alpha * wTw + beta * TSE), wTw

def grad_weights(model):

    g = torch.cat([model.fc_1.weight.grad.view(-1), model.fc_1.bias.grad.view(-1), 
                   model.fc_2.weight.grad.view(-1), model.fc_2.bias.grad.view(-1), 
                   model.fc_3.weight.grad.view(-1), model.fc_3.bias.grad.view(-1)])
    return g

In [134]:
model = Network(len(feature_headings))
optim = Adam(model.parameters())
alpha = 1.
beta = 1.

batch_size = 64
NUM_EPOCHS = 20
train_dataset = TensorDataset(x_train, y_train)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size)

for epoch in range(NUM_EPOCHS):

    print(f"Starting Epoch {epoch + 1} --")

    for batch, xy_batch in enumerate(train_dataloader):

        x_batch, y_batch = xy_batch
        y_hat = model(x_batch).squeeze()
        lpw, wTw = log_posterior_weights(model, y_hat, y_batch, alpha, beta)

        optim.zero_grad()
        (-lpw).backward()
        optim.step()

    y_hat = model(x_train).squeeze()

    H = 0
    for idx in range(y_hat.size(0)):
        model.zero_grad()
        y_hat[idx].backward(retain_graph=True)
        g = grad_weights(model)
        H += g * g[..., None]

        

Starting Epoch 1 --
Starting Epoch 2 --
Starting Epoch 3 --
Starting Epoch 4 --
Starting Epoch 5 --
Starting Epoch 6 --
Starting Epoch 7 --
Starting Epoch 8 --
Starting Epoch 9 --
Starting Epoch 10 --
Starting Epoch 11 --
Starting Epoch 12 --
Starting Epoch 13 --
Starting Epoch 14 --
Starting Epoch 15 --
Starting Epoch 16 --
Starting Epoch 17 --
Starting Epoch 18 --
Starting Epoch 19 --
Starting Epoch 20 --
